In [2]:
import sys

def return_prompt(sample, qa_index=0):
    sys.path.insert(0, "/home/srini/time_series")
    from loopqa.eval import LoopQAEvaluator

    txt = LoopQAEvaluator.format_patient_context(sample['input_context'])
    qa_item = sample['qa_pairs'][qa_index]
    prompt = LoopQAEvaluator.create_prompt(
        txt,
        qa_item['question_text'],
        qa_item['answer_instruction'],
        qa_item['answer_type'],
        example_answer=qa_item['example_answer'],
    )
    return prompt


In [3]:
import numpy as np
import json
import pandas as pd
import random
def format_time_info(mins):
    mins = float(mins)
    day = int(mins // 1440) + 1  # ← day 1 to 14, no weekly reset
    time_of_day = mins % 1440
    hours = int(time_of_day // 60)
    minutes = int(time_of_day % 60)
    time_str = f"{hours:02d}:{minutes:02d}" 
    return day, time_str


df = pd.read_csv("insulin_input_normal.csv")

df = df.T

sample_patient = random.randint(0,19)
with open("./ablation_questions/questions_and_answers_0.jsonl", "w") as h:
#loop over all 20 patients
    for i in range(0, 20):
        file_path = f"/home/srini/time_series/py-mgipsim/SimulationData/normal_day/normal_day_{i}_simulation_data.jsonl"


        #count for each time period for each patient
        dic = {
        "morning": 0,
        "afternoon": 0, 
        "evening": 0,
        "night": 0
        }

        with open(file_path, "r") as f:
            data = json.load(f)
        
        time = [t * 5 for t in range(len(data['bg_mgdl']))]
        carb_events = []
        running_events = []
        cycling_events = []
        cut_idx = 7 * 288
        bg_values = data['bg_mgdl'][0:cut_idx]
        insulin_values = df[i].to_list()[:cut_idx]
        cut_time = time[cut_idx]
        bg_values_per_day = np.array_split(bg_values, 7)
        
        assert len(bg_values_per_day[0]) == 288, "bg_values_per_day[0] is not equal to 288"

        for ex in data['exercise_events']:
            if ex['exercise_type'] == 'running':
                if ex['time'] < cut_time: 
                    running_events.append(ex)
            elif ex['exercise_type'] == 'cycling':
                if ex['time'] < cut_time: 
                    cycling_events.append(ex)

        for carb in data['carb_events']:
            if carb['time'] < cut_time:
                carb_events.append(carb)


        # Find the time of the day when the bg_values are highest for each day
        
        for day, day_data in enumerate(bg_values_per_day):
            max_mins = np.argmax(day_data)
            max_time_of_day = ( max_mins * 5 )
            if 360 <= max_time_of_day < 720:
                period = "Morning"
            elif 720 <= max_time_of_day < 1080:
                period = "Afternoon"
            elif 1080 <= max_time_of_day < 1440:
                period = "Evening"
            else:
                period = "Night"
            
            if period == "Morning":
                dic["morning"] += 1
            elif period == "Afternoon":
                dic["afternoon"] += 1
            elif period == "Evening":
                dic["evening"] += 1
            else:
                dic["night"] += 1

            #print(f"Day {day+1}: Highest BG at {format_time_info(max_mins * 5)[1]} ({period})")


        # Find the most common time period
        most_common_period = max(dic, key=dic.get)
        print("For Patient", i , "max glucose occurs in" , most_common_period)
        
        example_answer = ""
        for key in dic.keys():
            if key != most_common_period:
                example_answer = key
                break
        

        output_json = {}
        output_json['patient_id'] = 'Patient_'+str(i)
        output_json['input_context'] = {}
        output_json['input_context']['running_events'] = running_events
        output_json['input_context']['cycling_events'] = cycling_events
        output_json['input_context']['insulin_events'] = insulin_values
        output_json['input_context']['carb_events'] = carb_events
        output_json['input_context']['bg_mgdl'] = bg_values
        output_json['qa_pairs'] = []
        assert len(bg_values) == len(insulin_values), "bg_values and insulin_values have different lengths"
        
        qa = {}
        qa['patient_id'] = 'Patient_'+str(i)
        qa['function_name'] = 'get_bg_values'
        qa['question_id'] = 'pm_0'
        qa['question_text'] = 'Predict during what time of the day will my blood glucose level be highest?'
        qa['answer_generation_rule'] = "Use the sampled blood glucose values to find the time of day when the glucose level is highest for each day, where morning is 6AM–12PM, afternoon is 12PM–6PM, evening is 6PM–12AM, and night is 12AM–6AM. Return the most common time period based on the counts across all days."
        qa['answer_instruction'] = "Use the sampled blood glucose values to predict the time of the day when the blood glucose level will be highest for each day and then return the most common time period based on the count of each time period"
        qa['answer_type'] = "string"
        qa['metric'] = 'accuracy'
        qa['answer_instruction'] = qa['answer_generation_rule']
        
        
        
        assert most_common_period != example_answer, "most_common_period and example_answer are the same"
        qa['answer'] = most_common_period
        qa['example_answer'] = example_answer
        output_json['qa_pairs'].append(qa)

        if i == sample_patient:
            print("sample patient", i)
            prompt = return_prompt(output_json)
            with open("./ablation_prompts/0.txt", "w") as f:
                f.write(prompt)
                
        h.write(json.dumps(output_json) + "\n")


For Patient 0 max glucose occurs in morning
For Patient 1 max glucose occurs in afternoon
For Patient 2 max glucose occurs in morning
For Patient 3 max glucose occurs in morning
For Patient 4 max glucose occurs in evening
For Patient 5 max glucose occurs in evening
For Patient 6 max glucose occurs in morning
For Patient 7 max glucose occurs in morning
For Patient 8 max glucose occurs in morning
For Patient 9 max glucose occurs in morning
For Patient 10 max glucose occurs in evening
For Patient 11 max glucose occurs in morning
For Patient 12 max glucose occurs in morning
For Patient 13 max glucose occurs in morning
For Patient 14 max glucose occurs in morning
For Patient 15 max glucose occurs in morning
For Patient 16 max glucose occurs in afternoon
sample patient 16
For Patient 17 max glucose occurs in morning
For Patient 18 max glucose occurs in morning
For Patient 19 max glucose occurs in morning


### 2. cycling or running

In [4]:
# transpose the dataframe
import pandas as pd
import json
import random
import numpy as np

def calc(bg_diff_cycling, bg_diff_running):
    if bg_diff_cycling > 0 and bg_diff_running > 0:
        return "cycling" if bg_diff_cycling < bg_diff_running else "running"
    elif bg_diff_cycling > 0 and bg_diff_running < 0:
        return "running"
    elif bg_diff_cycling < 0 and bg_diff_running > 0:
        return "cycling"
    else:
        return "cycling" if bg_diff_cycling < bg_diff_running else "running"

def format_time_info(mins):
    mins = float(mins)
    day = int(mins // 1440) + 1  # ← day 1 to 14, no weekly reset
    time_of_day = mins % 1440
    hours = int(time_of_day // 60)
    minutes = int(time_of_day % 60)
    time_str = f"{hours:02d}:{minutes:02d}" 
    return day, time_str

df = pd.read_csv("insulin_input_normal.csv")
df = df.T

sample_patient = random.randint(0,19)

with open('./ablation_questions/questions_and_answers_9.jsonl', 'w') as h:
    for i in range(0,20):
        file_path = f"/home/srini/time_series/py-mgipsim/SimulationData/normal_day/normal_day_{i}_simulation_data.jsonl"
        with open(file_path, "r") as f:
            data = json.load(f)

        bg_values = data['bg_mgdl']
        bg_time = [t * 5 for t in range(len(bg_values))]

        insulin_values = df[i].to_list()
        running_events = []
        cycling_events = []
        carb_events = []

        output_json = {}
        output_json['patient_id'] = 'Patient_'+str(i)
        output_json['input_context'] = {}
        
        main_idx = 7 * 288
        main_time = bg_time[main_idx]
        

        cut_idx_cycling = 7 * 288 + 216
        cut_time_cycling = bg_time[cut_idx_cycling]
        
        cut_idx_running = 7 * 288 + 84
        cut_time_running = bg_time[cut_idx_running]
        
        
        for cycling in data['exercise_events']:
            if cycling['exercise_type'] == 'cycling':
                if cycling['time'] >= cut_time_cycling - 120 and cycling['time'] <= cut_time_cycling:
                    cycling_time = (cycling['time']//5) + 1

        for running in data['exercise_events']:
            if running['exercise_type'] == 'running':
                if running['time'] >= cut_time_running - 120 and running['time'] <= cut_time_running:
                    running_time = (running['time']//5) + 1

        cut_idx_cycling = int(cycling_time)
        end_idx_cycling = int(cycling_time) + 4 #cycle for 20 mins and check glucose level 

        #cut_time_cycling = bg_time[cut_idx_cycling]

        bg_diff_cycling = bg_values[end_idx_cycling] - bg_values[cut_idx_cycling]
        
        cut_idx_running = int(running_time)
        end_idx_running = int(running_time) + 6 #30 minutes after running starts
        cut_time_running = bg_time[cut_idx_running]
        bg_diff_running = bg_values[end_idx_running] - bg_values[cut_idx_running]
        #print("running time", format_time_info(bg_time[cut_idx_running]))
        #print("cycling time", format_time_info(bg_time[cut_idx_cycling]))
        print("bg_diff_cycling", bg_diff_cycling)
        print("bg_diff_running", bg_diff_running)

        better_option = calc(bg_diff_cycling, bg_diff_running)
        
        print("better_option", better_option)
        print("************************************************")
        count=0
        for ex in data['exercise_events']:
            if ex['exercise_type'] == 'running':
                if ex['time'] <= main_time: 
                    running_events.append(ex)
            elif ex['exercise_type'] == 'cycling':
                if ex['time'] <= main_time: 
                    cycling_events.append(ex)

        for carb in data['carb_events']:
            if carb['time'] <= main_time:
                carb_events.append(carb)
        
        #print(running_events[-1])
        #print(cycling_events[-1])
        qa = {}
        qa['patient_id'] = 'Patient_'+str(i)
        qa['function_name'] = 'check_better_exercise'
        qa['question_id'] = 'pm_9'
        qa['question_text'] = 'Which exercise is better for me to bring down blood sugar tomorrow - cycling or running?'
        qa['answer_generation_rule'] = "Assume cycling is performed for 20 minutes and running is performed for 30 minutes. Estimate the effect of each exercise type on lowering blood sugar and compare them. Return the exercise type that reduces blood sugar the most."
        qa['answer_instruction'] = "If you think cycling is better, return 'cycling', otherwise return 'running'"
        qa['answer_type'] = "string"
        qa['metric'] =  'accuracy'
        qa['example_answer'] = "cycling" if better_option == "running" else "running"
        qa['answer'] = better_option
        qa['answer_instruction'] = qa['answer_generation_rule']
        output_json['input_context']['running_events'] = running_events
        output_json['input_context']['cycling_events'] = cycling_events
        output_json['input_context']['carb_events'] = carb_events
        output_json['input_context']['insulin_events'] = df[i].to_list()[:main_idx]
        output_json['input_context']['bg_mgdl'] = data['bg_mgdl'][0:main_idx]
        output_json['qa_pairs'] = []
        output_json['qa_pairs'].append(qa)

        assert qa['example_answer'] != qa['answer']
        if i == sample_patient:
            print("sample patient", i)
            prompt = return_prompt(output_json)
            with open("./ablation_prompts/questions_and_answers_9.txt", "w") as f:
                f.write(prompt)
        h.write(json.dumps(output_json) + "\n")

bg_diff_cycling -36.015396092300676
bg_diff_running -17.19543087649852
better_option cycling
************************************************
bg_diff_cycling -2.8039438648186774
bg_diff_running 0.2867825012386689
better_option cycling
************************************************
bg_diff_cycling -17.679905458490765
bg_diff_running -2.6333632034675105
better_option cycling
************************************************
bg_diff_cycling -3.2448269429873307
bg_diff_running 20.76834839896425
better_option cycling
************************************************
bg_diff_cycling -39.26806779744686
bg_diff_running -0.004154722479142947
better_option cycling
************************************************
bg_diff_cycling -17.731565369038947
bg_diff_running -50.836006401570266
better_option running
************************************************
bg_diff_cycling -14.599641828882298
bg_diff_running -45.08827795432693
better_option running
************************************************
bg_

In [5]:
def max_glucose_slope(bg_values, start_idx, end_idx):
    max_slope = float('-inf')
    for i in range(start_idx + 1 , end_idx + 1):
        slope = bg_values[i] - bg_values[i-1]  # 5-minute step
        if slope > max_slope:
            max_slope = slope
    return max_slope



# transpose the dataframe
import pandas as pd
import json
import random
import numpy as np

def format_time_info(mins):
    mins = float(mins)
    day = int(mins // 1440) + 1  # ← day 1 to 14, no weekly reset
    time_of_day = mins % 1440
    hours = int(time_of_day // 60)
    minutes = int(time_of_day % 60)
    time_str = f"{hours:02d}:{minutes:02d}" 
    return day, time_str

df = pd.read_csv("insulin_input_normal.csv")
df = df.T
i = 0

sample_patient = random.randint(0,19)


with open('./ablation_questions/questions_and_answers_17.jsonl', 'w') as h:
    for i in range(0,20):
        file_path = f"/home/srini/time_series/py-mgipsim/SimulationData/normal_day/normal_day_{i}_simulation_data.jsonl"
        with open(file_path, "r") as f:
            data = json.load(f)

        bg_values = data['bg_mgdl']
        bg_time = [t * 5 for t in range(len(bg_values))]

        cut_idx = 7 * 288
        cut_time = bg_time[cut_idx]
        
        cut_time_start_lunch = 7 * 288 +  108
        cut_time_end_lunch = 7 * 288 + 156

        cut_time_start_dinner = 7 * 288 + 216
        cut_time_end_dinner = 7 * 288 + 264

        lunch_slope = max_glucose_slope(bg_values, cut_time_start_lunch, cut_time_end_lunch)
        dinner_slope = max_glucose_slope(bg_values, cut_time_start_dinner, cut_time_end_dinner)

    
        ans = ''
        if lunch_slope > dinner_slope:
            ans = 'Lunch'
        else:
            ans = 'Dinner'


        
        running_events = []
        cycling_events = []
        carb_events = []

        for ex in data['exercise_events']:
            if ex['exercise_type'] == 'running':
                if ex['time'] < cut_time: 
                    running_events.append(ex)
            elif ex['exercise_type'] == 'cycling':
                if ex['time'] < cut_time: 
                    cycling_events.append(ex)
                    
        for carb in data['carb_events']:
            if carb['time'] < cut_time:
                carb_events.append(carb)

        output_json = {}
        output_json['patient_id'] = 'Patient_'+str(i)
        output_json['input_context'] = {}

        qa = {}
        qa['patient_id'] = 'Patient_'+str(i)
        qa['function_name'] = 'predict_increase'
        qa['question_id'] = 'pm_17'
        qa['question_text'] = 'Predict if my blood sugar will go up faster during lunch time (9AM-1PM) or dinner time (6PM-10PM) tomorrow?'
        qa['answer_generation_rule'] = "Predict whether the adjacent time periods with the highest glucose change occur during lunch (9 AM–1 PM) or dinner (6 PM–10 PM). Return 'Lunch' if blood sugar rises faster during lunch time, or 'Dinner' if it rises faster during dinner time."
        qa['answer_instruction'] = "Return a string value of 'lunch' or 'dinner' where 'lunch' means that the blood sugar will go up faster during lunch time (9-1PM) and 'dinner' means that the blood sugar will go up faster during dinner time (6-10PM)."
        qa['answer_type'] = "string"
        qa['metric'] =  'Accuracy'
        qa['example_answer'] = 'Lunch' if ans == 'Dinner' else 'Dinner'
        qa['answer'] = ans
        qa['answer_instruction'] = qa['answer_generation_rule']
        print("ans", ans)

        assert qa['example_answer'] != ans , "Example answer and original answer are the same"
        output_json['input_context']['running_events'] = running_events
        output_json['input_context']['cycling_events'] = cycling_events
        output_json['input_context']['carb_events'] = carb_events
        output_json['input_context']['insulin_events'] = df[i].to_list()[:cut_idx]
        output_json['input_context']['bg_mgdl'] = data['bg_mgdl'][0:cut_idx]
        output_json['qa_pairs'] = []
        output_json['qa_pairs'].append(qa)

        assert qa['example_answer'] != qa['answer'] 
        if i == sample_patient:
            print("sample patient", i)
            prompt = return_prompt(output_json)
            with open("./ablation_prompts/16.txt", "w") as f:
                f.write(prompt)
        
        h.write(json.dumps(output_json) + "\n")

ans Lunch
ans Lunch
ans Lunch
ans Dinner
ans Lunch
ans Lunch
ans Lunch
ans Lunch
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
ans Dinner
sample patient 19


In [ ]:
import json
dic = {}
with open("/home/srini/time_series/py-mgipsim/questions/combined_prediction.jsonl" , "r") as f:
    for line in f:
        data = json.loads(line)
        question = data['qa_pairs'][0]['question_text']
        dic[question] = 1

## Combine all

In [ ]:
import json

final_output = {}
c = 0
with open("/home/srini/time_series/py-mgipsim/ablation_questions/ablation_prediction.jsonl" , "w") as g:
    for i in range(0, 21):
        try:
            with open(f"/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_{i}.jsonl"  , "r") as f:
                for line in f:
                    data = json.loads(line)
                    g.write(line)
                c += 1
        except Exception as e:
            print(e)
            pass
        

[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_1.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_2.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_3.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_4.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_5.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_6.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_7.jsonl'
[Errno 2] No such file or directory: '/home/srini/time_series/py-mgipsim/ablation_questions/questions_and_answers_8.jsonl'
[Errno 2] No suc

In [7]:
import json
dic = {}
with open("/home/srini/time_series/py-mgipsim/ablation_questions/ablation_prediction.jsonl" , "r") as f:
    for line in f:
        data = json.loads(line)
        question = data['qa_pairs'][0]['question_text']
        dic[question] = 1

In [9]:
dic.keys()

dict_keys(['Predict during what time of the day will my blood glucose level be highest?', 'Which exercise is better for me to bring down blood sugar tomorrow - cycling or running?', 'Predict if my blood sugar will go up faster during lunch time (9AM-1PM) or dinner time (6PM-10PM) tomorrow?'])